# AnyCam — 05c · Step 7 round 3: final SVCT configuration

Rounds 1-2 showed the consistency loss hurts and the best model was still improving at 80 epochs. Final setup:
- no consistency loss (one virtual camera per batch, full 204 bands 30% of the time), calibration errors 50%
- more phone-like practice cameras (45% rgb-like, 35% multispectral, 20% random); the 7 real cameras stay held out
- 150 epochs (patience 50)

Runs (~2.5 h): **SVCT final** (wavelength layer) and the clean ablation **same without wavelength layer**.
Setup: **HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On**. GitHub: **ShahnawazQ007/one-model-many-camera**,
file `kaggle_runs/05c_step7_round3.ipynb`.

In [ ]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

In [ ]:
# 1) SVCT final (ours)
!python experiments/train_svct.py --data {DATA} --stem hyper --views 1 --p-full 0.3 --radio 0.5 --sampler mixed --family-weights 0.45,0.35,0.2 --select acc --epochs 150 --patience 50 --out runs/svct_final

In [ ]:
# 2) clean ablation: identical, but no wavelength layer (bands interpolated back to 204)
!python experiments/train_svct.py --data {DATA} --stem interp --views 1 --p-full 0.3 --radio 0.5 --sampler mixed --family-weights 0.45,0.35,0.2 --select acc --epochs 150 --patience 50 --out runs/svct_final_interp

In [ ]:
!mkdir -p runs/oracles && cp -r results/step07_svct_round2/ runs/r2 2>/dev/null; python - <<'PY'
import json, pathlib
for f in pathlib.Path('results/step07_svct_round2').glob('oracle_*_metrics.json'):
    d = pathlib.Path('runs/oracles') / f.name.replace('_metrics.json', ''); d.mkdir(parents=True, exist_ok=True)
    (d / 'metrics.json').write_text(f.read_text())
PY
!python scripts/plot_step7.py --baseline results/step06_camera_test/camera_eval.csv --oracles "runs/oracles/oracle_*" --runs "SVCT final (ours)=runs/svct_final" "no wavelength layer=runs/svct_final_interp" --out runs/step7_final
from IPython.display import Image, display
display(Image('runs/step7_final/step7_compare.png'))
!for f in runs/svct_final*/camera_eval.csv; do echo "### $f"; cat $f; done

## Done
**Save Version → Quick Save** → GitHub repository **ShahnawazQ007/one-model-many-camera**, file
`kaggle_runs/05c_step7_round3.ipynb`. Stop the session (power button). Tell Claude "done".